# Izdelava značilk

Naložimo podatke

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RANDOM_STATE = 42

df = pd.read_csv("./data/processed/matches_clean.csv")

df["tourney_date"] = pd.to_datetime(df["tourney_date"])
df["year"] = df["tourney_date"].dt.year

df = df.sort_values(["tourney_date", "tourney_id", "match_num"]).reset_index(drop=True)

df.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,loser_ioc,loser_age,best_of,round,winner_rank,winner_rank_points,loser_rank,loser_rank_points,source_year,year
0,2000-339,Adelaide,Hard,32,A,2000-01-03,1,102358,1.0,NaN,...,FRA,22.0,3,R32,4.0,2606.0,56.0,805.0,2000,2000
1,2000-339,Adelaide,Hard,32,A,2000-01-03,2,103819,NaN,NaN,...,GER,24.8,3,R32,64.0,749.0,91.0,525.0,2000,2000
2,2000-339,Adelaide,Hard,32,A,2000-01-03,3,102998,NaN,NaN,...,AUS,28.7,3,R32,58.0,803.0,105.0,449.0,2000,2000
3,2000-339,Adelaide,Hard,32,A,2000-01-03,4,103206,7.0,NaN,...,AUS,23.6,3,R32,27.0,1298.0,54.0,845.0,2000,2000
4,2000-339,Adelaide,Hard,32,A,2000-01-03,5,102796,3.0,NaN,...,AUS,25.5,3,R32,15.0,1748.0,154.0,297.0,2000,2000


### Sanitizacija stolpcev

Izberemo stolpce ki jih bomo uporabili

In [2]:
base_cols = [
    # identifikacija tekme
    "tourney_id",
    "tourney_name",
    "tourney_date",
    "year",
    "match_num",

    # turnir
    "surface",
    "tourney_level",
    "best_of",
    "round",
    "draw_size",

    # igralca
    "winner_id",
    "winner_name",
    "winner_hand",
    "winner_ht",
    "winner_age",
    "winner_rank",
    "winner_rank_points",

    "loser_id",
    "loser_name",
    "loser_hand",
    "loser_ht",
    "loser_age",
    "loser_rank",
    "loser_rank_points",
]

# brez:
    # winner_seed
    # winner_entry
    # winner_ioc
    # loser_seed
    # loser_entry
    # loser_ioc
    # source_year

df_base = df[base_cols].copy()
df_base.shape

(68779, 24)

Odstranimo vrstice ki nimajo potrebnih podatkov

In [3]:
required_cols = [
    "surface",
    "tourney_level",
    "best_of",
    "round",
    "winner_id",
    "loser_id",
    "winner_rank",
    "loser_rank",
    "winner_age",
    "loser_age",
    "winner_rank_points",
    "loser_rank_points",
]

before = len(df_base)

df_base = df_base.dropna(subset=required_cols).copy()

after = len(df_base)

print(f"Odstranjenih vrstic: {before - after}")
print(f"Ostalo vrstic: {after}")

Odstranjenih vrstic: 0
Ostalo vrstic: 68779


### Sprememba winner/loser v p1/p2

Naključno določimo ali zmaga p1 ali p2

In [4]:
rng = np.random.default_rng(RANDOM_STATE)

features = df_base.copy()

features["p1_is_winner"] = rng.random(len(features)) < 0.5
features["target"] = features["p1_is_winner"].astype(int)

features[["winner_name", "loser_name", "p1_is_winner", "target"]].head()

,winner_name,loser_name,p1_is_winner,target
0,Thomas Enqvist,Arnaud Clement,False,0
1,Roger Federer,Jens Knippschild,True,1
2,Jan Michael Gambill,Wayne Arthurs,False,0
3,Sebastien Grosjean,Andrew Ilie,False,0
4,Magnus Norman,Scott Draper,True,1


In [5]:
def choose_p1(row, winner_col, loser_col):
    return row[winner_col] if row["p1_is_winner"] else row[loser_col]

def choose_p2(row, winner_col, loser_col):
    return row[loser_col] if row["p1_is_winner"] else row[winner_col]

Vpisemo ustreznega igralca na mesti p1 in p2

In [6]:
player_attrs = {
    "id": ("winner_id", "loser_id"),
    "name": ("winner_name", "loser_name"),
    "hand": ("winner_hand", "loser_hand"),
    "height": ("winner_ht", "loser_ht"),
    "age": ("winner_age", "loser_age"),
    "rank": ("winner_rank", "loser_rank"),
    "rank_points": ("winner_rank_points", "loser_rank_points"),
}

for attr, (winner_col, loser_col) in player_attrs.items():
    features[f"p1_{attr}"] = np.where(
        features["p1_is_winner"],
        features[winner_col],
        features[loser_col]
    )

    features[f"p2_{attr}"] = np.where(
        features["p1_is_winner"],
        features[loser_col],
        features[winner_col]
    )

Target 1 pomeni da je zmagal p1, target 0 pa pomeni da p1 ni zmagal.

In [7]:
features[
    [
        "winner_name", "loser_name",
        "p1_name", "p2_name",
        "target"
    ]
].head(10)

,winner_name,loser_name,p1_name,p2_name,target
0,Thomas Enqvist,Arnaud Clement,Arnaud Clement,Thomas Enqvist,0
1,Roger Federer,Jens Knippschild,Roger Federer,Jens Knippschild,1
2,Jan Michael Gambill,Wayne Arthurs,Wayne Arthurs,Jan Michael Gambill,0
3,Sebastien Grosjean,Andrew Ilie,Andrew Ilie,Sebastien Grosjean,0
4,Magnus Norman,Scott Draper,Magnus Norman,Scott Draper,1
5,Richard Fromberg,Todd Woodbridge,Todd Woodbridge,Richard Fromberg,0
6,James Sekulov,Gianluca Pozzi,Gianluca Pozzi,James Sekulov,0
7,Alberto Martin,Vincent Spadea,Vincent Spadea,Alberto Martin,0
8,Lleyton Hewitt,Mark Woodforde,Lleyton Hewitt,Mark Woodforde,1
9,Dejan Petrovic,Stephane Huet,Dejan Petrovic,Stephane Huet,1


Ustvarimo razlike med igralcema

In [8]:
features["rank_diff"] = features["p1_rank"] - features["p2_rank"]
features["rank_points_diff"] = features["p1_rank_points"] - features["p2_rank_points"]
features["age_diff"] = features["p1_age"] - features["p2_age"]
features["height_diff"] = features["p1_height"] - features["p2_height"]

In [9]:
height_before = features[["p1_height", "p2_height"]].isna().sum()

Če višine ni, pogledamo v datoteko atp_players.csv, drugače vzamemo povprečno višino

In [10]:
players = pd.read_csv("./data/atp_players.csv")

players_height = players[["player_id", "height"]].copy()
players_height = players_height.rename(columns={"height": "player_height"})

players_height.head()

/var/folders/gw/1878dlp13hx3km8s7zyfvht40000gn/T/ipykernel_3726/2274903089.py:1: DtypeWarning: Columns (0: wikidata_id) have mixed types. Specify dtype option on import or set low_memory=False.
  players = pd.read_csv("./data/atp_players.csv")


,player_id,player_height
0,100001,185.0
1,100002,168.0
2,100003,180.0
3,100004,NaN
4,100005,188.0


In [11]:
features = features.merge(
    players_height.rename(columns={
        "player_id": "p1_id",
        "player_height": "p1_height_from_players"
    }),
    on="p1_id",
    how="left"
)

features = features.merge(
    players_height.rename(columns={
        "player_id": "p2_id",
        "player_height": "p2_height_from_players"
    }),
    on="p2_id",
    how="left"
)

In [12]:

features["p1_height"] = features["p1_height"].fillna(features["p1_height_from_players"])
features["p2_height"] = features["p2_height"].fillna(features["p2_height_from_players"])

features["p1_height"] = features["p1_height"].fillna(features["p1_height"].median())
features["p2_height"] = features["p2_height"].fillna(features["p2_height"].median())

features["height_diff"] = features["p1_height"] - features["p2_height"]

Odstranimo pomozna stolpca

In [13]:
features = features.drop(columns=[
    "p1_height_from_players",
    "p2_height_from_players"
    
])

Naredimo značilko za roke in kategoricno znacilko za round spremenimo v ordinalno

In [14]:
features["p1_hand"] = features["p1_hand"].fillna("U")
features["p2_hand"] = features["p2_hand"].fillna("U")

features["hand_matchup"] = features["p1_hand"] + "_vs_" + features["p2_hand"]

features["hand_matchup"].value_counts()

hand_matchup
R_vs_R    51947
L_vs_R     7822
R_vs_L     7760
L_vs_L     1137
U_vs_R       50
R_vs_U       46
U_vs_L        9
L_vs_U        6
L_vs_A        1
R_vs_A        1
Name: count, dtype: int64

In [15]:
round_order = {
    "R128": 1,
    "R64": 2,
    "R32": 3,
    "R16": 4,
    "QF": 5,
    "SF": 6,
    "F": 7,
    "RR": 4,
    "ER": 5,
    "BR": 7,
}

features["round_num"] = features["round"].map(round_order)

Naredimo končni nabor značilk in ustvarimo dataset

In [16]:
metadata_cols = [
    "tourney_id",
    "tourney_name",
    "tourney_date",
    "year",
    "match_num",
    "p1_id",
    "p1_name",
    "p2_id",
    "p2_name",
    "target",
]

categorical_features = [
    "surface",
    "tourney_level",
    "best_of",
    "round",
    "hand_matchup",
]

numeric_features = [
    "rank_diff",
    "rank_points_diff",
    "age_diff",
    "height_diff",
    "round_num",
    "draw_size",
]

In [17]:
final_cols = metadata_cols + categorical_features + numeric_features

features_basic = features[final_cols].copy()

features_basic.head()

,tourney_id,tourney_name,tourney_date,year,match_num,p1_id,p1_name,p2_id,p2_name,target,...,tourney_level,best_of,round,hand_matchup,rank_diff,rank_points_diff,age_diff,height_diff,round_num,draw_size
0,2000-339,Adelaide,2000-01-03,2000,1,103096,Arnaud Clement,102358,Thomas Enqvist,0,...,A,3,R32,R_vs_R,52.0,-1801.0,-3.7,-17.0,3,32
1,2000-339,Adelaide,2000-01-03,2000,2,103819,Roger Federer,102533,Jens Knippschild,1,...,A,3,R32,R_vs_R,-27.0,224.0,-6.5,-5.0,3,32
2,2000-339,Adelaide,2000-01-03,2000,3,101885,Wayne Arthurs,102998,Jan Michael Gambill,0,...,A,3,R32,L_vs_R,47.0,-354.0,6.2,0.0,3,32
3,2000-339,Adelaide,2000-01-03,2000,4,102776,Andrew Ilie,103206,Sebastien Grosjean,0,...,A,3,R32,R_vs_R,27.0,-453.0,2.1,5.0,3,32
4,2000-339,Adelaide,2000-01-03,2000,5,102796,Magnus Norman,102401,Scott Draper,1,...,A,3,R32,R_vs_L,-139.0,1451.0,-2.0,10.0,3,32


Preverimo če je target 1 proti 0 pribljizno 50/50

In [18]:
features_basic["target"].value_counts(normalize=True)

target
0    0.502145
1    0.497855
Name: proportion, dtype: float64

rank_diff in target naj bi bila negativno povezana

In [19]:
features_basic[["rank_diff", "target"]].corr()

,rank_diff,target
rank_diff,1.000000,-0.245175
target,-0.245175,1.000000


Shranimo dataset v osnovne_znacilke.csv

In [20]:
output_path = Path("./data/processed/osnovne_znacilke.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

features_basic.to_csv(output_path, index=False)

print(f"Saved to {output_path}")
print(features_basic.shape)

Saved to data/processed/osnovne_znacilke.csv
(68779, 21)
